In [16]:
import pandas as pd
import unicodedata
from pathlib import Path

In [17]:
# =========================
# CONFIG: rutas de archivos
# =========================
FILE_2018 = Path("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/scrapin_onpe__2018.xlsx")
FILE_2022 = Path("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/scrapin_onpe__2022.xlsx")
FILE_MAESTRO = Path("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_distrital.xlsx")

OUT_2018 = Path("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/scrapin_onpe__2018_con_votos_orden.xlsx")
OUT_2022 = Path("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/scrapin_onpe__2022_con_votos_orden.xlsx")

REPORT_UNMATCHED_2018 = Path("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/unmatched_2018.xlsx")
REPORT_UNMATCHED_2022 = Path("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/unmatched_2022.xlsx")

In [18]:
# ======================================
# Helpers: normalización de texto / keys
# ======================================
def _strip_accents(s: str) -> str:
    """Remueve tildes/diacríticos."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = str(s)
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", s)
        if not unicodedata.combining(ch)
    )

def norm_org(s: str) -> str:
    """
    Normaliza nombres de organización política:
    - upper
    - sin tildes
    - espacios colapsados
    """
    s = _strip_accents(s).upper().strip()
    s = " ".join(s.split())
    return s

def to_int_safe(x):
    """Convierte a int si se puede, si no devuelve NA."""
    try:
        if pd.isna(x):
            return pd.NA
        return int(x)
    except Exception:
        return pd.NA

In [19]:
# ======================================
# Cargar y preparar el maestro de votos
# ======================================
def load_master(path: Path) -> pd.DataFrame:
    master = pd.read_excel(path)

    # columnas esperadas del maestro:
    # ['ubigeo','año','region','provincia','distrito','organizacion_politica','total_votos','orden_aparicion']
    master = master.rename(columns={
        "año": "anio",
        "organizacion_politica": "political_organization",
        "total_votos": "votos",
        "orden_aparicion": "orden"
    })

    master["anio"] = master["anio"].apply(to_int_safe).astype("Int64")
    master["ubigeo"] = master["ubigeo"].apply(to_int_safe).astype("Int64")

    master["org_key"] = master["political_organization"].apply(norm_org)

    # Nos quedamos con las columnas mínimas para el merge
    master_keyed = master[["anio", "ubigeo", "org_key", "votos", "orden"]].copy()

    # Por seguridad, si hubiera duplicados en el maestro para la misma llave,
    # nos quedamos con el primero (o podrías agregar una validación estricta).
    master_keyed = master_keyed.drop_duplicates(subset=["anio", "ubigeo", "org_key"])

    return master_keyed

In [21]:
# ======================================
# Cargar, preparar y mergear scrapin ONPE
# ======================================
def enrich_scrapin(scrapin_path: Path, anio: int, master_keyed: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = pd.read_excel(scrapin_path)

    # Asegurar año y ubigeo
    df["anio"] = anio
    df["district_code"] = df["district_code"].apply(to_int_safe).astype("Int64")

    # Normalizar organización política
    df["org_key"] = df["political_organization"].apply(norm_org)

    # Merge (left): traemos votos y orden a cada fila (candidato)
    out = df.merge(
        master_keyed,
        how="left",
        left_on=["anio", "district_code", "org_key"],
        right_on=["anio", "ubigeo", "org_key"]
    )

    # Limpieza de columnas auxiliares del merge
    out = out.drop(columns=["ubigeo"], errors="ignore")

    # Ordenar columnas: agrega votos/orden cerca de la organización (opcional)
    # Si prefieres no tocar el orden, comenta este bloque.
    cols = list(out.columns)
    # mover votos/orden después de political_organization si existen
    if "political_organization" in cols and "votos" in cols and "orden" in cols:
        cols.remove("votos"); cols.remove("orden")
        idx = cols.index("political_organization") + 1
        cols = cols[:idx] + ["votos", "orden"] + cols[idx:]
        out = out[cols]

    # Reporte de no-matcheados (para auditoría)
    unmatched = out[out["votos"].isna() | out["orden"].isna()].copy()

    # Para que el reporte sea útil: un resumen único por llave
    unmatched_summary = (
        unmatched.groupby(["anio", "district_code", "political_organization", "org_key"], dropna=False)
        .size()
        .reset_index(name="filas_sin_match")
        .sort_values(["filas_sin_match"], ascending=False)
    )

    return out, unmatched_summary


def main():
    master_keyed = load_master(FILE_MAESTRO)

    out_2018, unmatched_2018 = enrich_scrapin(FILE_2018, 2018, master_keyed)
    out_2022, unmatched_2022 = enrich_scrapin(FILE_2022, 2022, master_keyed)

    # Guardar resultados
    out_2018.to_excel(OUT_2018, index=False)
    out_2022.to_excel(OUT_2022, index=False)

    # Guardar reportes de no-matcheados
    unmatched_2018.to_excel(REPORT_UNMATCHED_2018, index=False)
    unmatched_2022.to_excel(REPORT_UNMATCHED_2022, index=False)

    # Métricas rápidas en consola
    def stats(name, df):
        total = len(df)
        ok = df["votos"].notna().sum() if "votos" in df.columns else 0
        print(f"{name}: total filas={total:,} | con votos={ok:,} | sin match={total-ok:,}")

    stats("2018", out_2018)
    stats("2022", out_2022)

    print("Archivos generados:")
    print(" -", OUT_2018)
    print(" -", OUT_2022)
    print("Reportes no-matcheados:")
    print(" -", REPORT_UNMATCHED_2018)
    print(" -", REPORT_UNMATCHED_2022)


if __name__ == "__main__":
    main()

2018: total filas=6,736 | con votos=6,433 | sin match=303
2022: total filas=4,653 | con votos=4,642 | sin match=11
Archivos generados:
 - /Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/scrapin_onpe__2018_con_votos_orden.xlsx
 - /Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/scrapin_onpe__2022_con_votos_orden.xlsx
Reportes no-matcheados:
 - /Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/unmatched_2018.xlsx
 - /Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/unmatched_2022.xlsx


In [32]:
import pandas as pd
import unicodedata
from pathlib import Path

# =========================
# Archivos de entrada/salida
# =========================
BASE_2018 = Path("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/scrapin_onpe__2018_con_votos_orden.xlsx")
BASE_2022 = Path("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/scrapin_onpe__2022_con_votos_orden.xlsx")

OUT_2022_TURNOVER = Path("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/scrapin_onpe__2022_con_votos_orden_turnover.xlsx")


In [33]:
# =========================
# Normalizadores
# =========================
def _strip_accents(s: str) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = str(s)
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", s)
        if not unicodedata.combining(ch)
    )

def norm_text(s: str) -> str:
    s = _strip_accents(s).upper().strip()
    s = " ".join(s.split())
    return s

def norm_org(s: str) -> str:
    return norm_text(s)

def pick_vote_col(df: pd.DataFrame) -> str:
    """
    Elige la mejor columna de votos para definir ganador.
    Ajusta la lista si tu dataset usa otro nombre.
    """
    candidates = [
        "votos_candidato",
        "candidate_votes",
        "votes_candidate",
        "votos",
    ]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(
        "No encuentro columna de votos. Esperaba alguna de: "
        + ", ".join(candidates)
    )

def pick_dni_col(df: pd.DataFrame) -> str:
    """
    Detecta la columna DNI (identificador de candidato).
    Ajusta la lista si tu dataset usa otro nombre.
    """
    candidates = [
        "dni", "DNI", "documento", "documento_identidad",
        "num_documento", "numero_documento", "id_dni", "id_candidato",
    ]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(
        "No encuentro columna DNI. Probé: " + ", ".join(candidates)
    )

def clean_dni_series(s: pd.Series) -> pd.Series:
    """
    Limpia DNI: string sin decimales, sin espacios. Mantiene NA.
    """
    s = s.copy()
    s = s.astype("string")
    s = s.str.replace(r"\.0$", "", regex=True)  # por si vino como float en Excel
    s = s.str.strip()
    s = s.replace({"": pd.NA, "NA": pd.NA, "NAN": pd.NA})
    return s


In [34]:
# =========================
# Ganadores por distrito/año
# =========================
def winners_by_district(df: pd.DataFrame, year: int) -> pd.DataFrame:
    """
    Devuelve 1 fila por distrito con el ganador (máximo votos):
    - org del ganador
    - dni del ganador
    """
    df = df.copy()

    if "district_code" not in df.columns:
        raise ValueError("No existe la columna 'district_code' en la base.")

    if "political_organization" not in df.columns:
        raise ValueError("No existe la columna 'political_organization' en la base.")

    vote_col = pick_vote_col(df)
    dni_col = pick_dni_col(df)

    df["org_key"] = df["political_organization"].apply(norm_org)
    df[dni_col] = clean_dni_series(df[dni_col])

    df[vote_col] = pd.to_numeric(df[vote_col], errors="coerce")

    winners = (
        df.sort_values(["district_code", vote_col], ascending=[True, False])
          .dropna(subset=[vote_col])
          .drop_duplicates(subset=["district_code"], keep="first")
          [["district_code", "org_key", dni_col]]
          .rename(columns={
              "org_key": f"winner_org_{year}",
              dni_col: f"winner_dni_{year}",
          })
          .reset_index(drop=True)
    )

    return winners, vote_col, dni_col

In [35]:
# =========================
# Main: crear turnover en 2022
# =========================
def main():
    df18 = pd.read_excel(BASE_2018)
    df22 = pd.read_excel(BASE_2022)

    win18, vote_col18, dni_col18 = winners_by_district(df18, 2018)
    win22, vote_col22, dni_col22 = winners_by_district(df22, 2022)

    out22 = df22.copy()

    out22["org_key"] = out22["political_organization"].apply(norm_org)
    out22[dni_col22] = clean_dni_series(out22[dni_col22])

    out22 = out22.merge(win18, on="district_code", how="left")
    out22 = out22.merge(win22, on="district_code", how="left")

    out22["turnover_org_2022"] = (
        out22["winner_org_2022"].notna() &
        out22["winner_org_2018"].notna() &
        (out22["winner_org_2022"] == out22["winner_org_2018"])
    ).astype(int)

    out22["turnover_candidato_2022"] = (
        out22["winner_dni_2022"].notna() &
        out22["winner_dni_2018"].notna() &
        (out22["winner_dni_2022"] == out22["winner_dni_2018"])
    ).astype(int)

    # 👉 ESTA LÍNEA VA DENTRO DE main(), alineada con las demás
    out22.to_excel(OUT_2022_TURNOVER, index=False)

    print("Archivo generado:", OUT_2022_TURNOVER)

In [36]:
if __name__ == "__main__":
    main()

Archivo generado: /Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/scrapin_onpe__2022_con_votos_orden_turnover.xlsx


In [39]:
import pandas as pd
from pathlib import Path

# =========================
# RUTAS (AJUSTA SI HACE FALTA)
# =========================
# Si estás en tu compu, usa nombres de archivo en la misma carpeta del notebook.
# Si estás en mi entorno, estas rutas /mnt/data funcionan.
BASE_2022 = Path("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/scrapin_onpe__2022_con_votos_orden_turnover.xlsx")
DENUNCIAS = Path("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/denuncias.xlsx")

# Alternativa si estás en mi entorno (descomenta):
# BASE_2022 = Path("/mnt/data/scrapin_onpe__2022_con_votos_orden_turnover.xlsx")
# DENUNCIAS = Path("/mnt/data/denuncias.xlsx")

OUT_FINAL = Path("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/base_2022_con_turnover_y_criminality_pre2018_2021.xlsx")

# =========================
# PARAMETROS: OPCION 1 (PRE-TRATAMIENTO)
# =========================
START_YEAR = 2018
END_YEAR = 2021

# =========================
# 1) Cargar base 2022
# =========================
df22 = pd.read_excel(BASE_2022)

if "district_code" not in df22.columns:
    raise ValueError("No encuentro 'district_code' en la base 2022. Revisa el nombre de tu columna UBIGEO.")

df22["district_code"] = pd.to_numeric(df22["district_code"], errors="coerce").astype("Int64")

# =========================
# 2) Cargar denuncias
# =========================
den = pd.read_excel(DENUNCIAS)

# Normalizar nombres (por si viene 'año')
if "año" in den.columns and "anio" not in den.columns:
    den = den.rename(columns={"año": "anio"})

needed = {"ubigeo", "anio", "cantidad"}
missing = needed - set(den.columns)
if missing:
    raise ValueError(f"En denuncias.xlsx faltan columnas: {missing}. Columnas disponibles: {list(den.columns)}")

den["ubigeo"] = pd.to_numeric(den["ubigeo"], errors="coerce").astype("Int64")
den["anio"] = pd.to_numeric(den["anio"], errors="coerce").astype("Int64")
den["cantidad"] = pd.to_numeric(den["cantidad"], errors="coerce")

# =========================
# 3) Filtrar 2018-2021 y promediar por ubigeo
# =========================
den_pre = den[(den["anio"] >= START_YEAR) & (den["anio"] <= END_YEAR)].copy()

complaints_avg = (
    den_pre.groupby("ubigeo", as_index=False)["cantidad"]
    .mean()
    .rename(columns={"cantidad": f"complaints_avg_{START_YEAR}_{END_YEAR}"})
)

# (Opcional) También el total del periodo, por si te lo piden
complaints_sum = (
    den_pre.groupby("ubigeo", as_index=False)["cantidad"]
    .sum()
    .rename(columns={"cantidad": f"complaints_sum_{START_YEAR}_{END_YEAR}"})
)

# =========================
# 4) Merge a la base 2022
# =========================
df22 = df22.merge(
    complaints_avg,
    how="left",
    left_on="district_code",
    right_on="ubigeo"
).drop(columns=["ubigeo"], errors="ignore")

df22 = df22.merge(
    complaints_sum,
    how="left",
    left_on="district_code",
    right_on="ubigeo"
).drop(columns=["ubigeo"], errors="ignore")

# Outcome en inglés (según te pidieron)
df22["criminality"] = df22[f"complaints_avg_{START_YEAR}_{END_YEAR}"]

# =========================
# 5) Guardar
# =========================
df22.to_excel(OUT_FINAL, index=False)

print("OK. Archivo final generado:", OUT_FINAL)
print("Outcome creado:", "criminality")
print("Columnas nuevas:",
      f"complaints_avg_{START_YEAR}_{END_YEAR}",
      f"complaints_sum_{START_YEAR}_{END_YEAR}")


OK. Archivo final generado: /Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/base_2022_con_turnover_y_criminality_pre2018_2021.xlsx
Outcome creado: criminality
Columnas nuevas: complaints_avg_2018_2021 complaints_sum_2018_2021
